# Week 7 Assignment — Delta Lake SCD Implementation

This notebook completes the Week 7 Delta Lake assignment using customer master and incremental data.

It covers:

1. Loading customer master and incremental datasets
2. Data cleaning and validation
3. SCD Type 1 logic
4. SCD Type 2 logic
5. Final output saving

**Important:** This notebook is made to run safely in VS Code/Jupyter. If Delta Spark cannot start because of Java/JAR/kernel issues, the notebook automatically runs a pandas fallback so the assignment outputs are still generated without breaking the notebook.

## 1. Setup Project Paths and Runtime

This cell prepares folders and tries to start a Delta-enabled Spark session. If Spark/Delta fails locally, the notebook continues using pandas mode.

In [1]:
from pathlib import Path
import sys
import os
import shutil
import traceback

print("Python executable used by this notebook:")
print(sys.executable)

# Make PySpark use the same Python environment as the notebook kernel
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Detect project root whether notebook is opened from root or notebooks folder
current_path = Path.cwd()
project_root = current_path.parent if current_path.name.lower() == "notebooks" else current_path

data_dir = project_root / "data"
delta_dir = project_root / "delta_tables"
output_dir = project_root / "outputs"

for folder in [data_dir, delta_dir, output_dir]:
    folder.mkdir(parents=True, exist_ok=True)

spark = None
DeltaTable = None
USE_DELTA = False
RUNTIME_MODE = "pandas_fallback"

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (
        col, trim, when, lit, sha2, concat_ws, to_date,
        count, sum as spark_sum
    )
    from delta import configure_spark_with_delta_pip
    from delta.tables import DeltaTable

    # Stop old Spark session if it exists
    try:
        spark.stop()
    except Exception:
        pass

    builder = (
        SparkSession.builder
        .appName("Week7-Delta-SCD-Assignment")
        .master("local[*]")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.driver.memory", "2g")
        .config("spark.executor.memory", "2g")
        .config("spark.sql.shuffle.partitions", "4")
        .config("spark.databricks.delta.schema.autoMerge.enabled", "true")
    )

    spark = configure_spark_with_delta_pip(builder).getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    USE_DELTA = True
    RUNTIME_MODE = "delta_spark"
    print("Delta Spark session started successfully.")
    print("Spark Version:", spark.version)

except Exception as e:
    print("Delta Spark could not start on this system.")
    print("Notebook will continue in pandas fallback mode, so execution does not stop.")
    print("Reason:", str(e)[:600])

print("Runtime Mode:", RUNTIME_MODE)
print("Project Root:", project_root)
print("Data Directory:", data_dir)
print("Delta Directory:", delta_dir)
print("Output Directory:", output_dir)

Python executable used by this notebook:
c:\Celebal Assignments\venv\Scripts\python.exe
Delta Spark could not start on this system.
Notebook will continue in pandas fallback mode, so execution does not stop.
Reason: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: C:\hadoop\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:432)
Runtime Mode: pandas_fallback
Project Root: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment
Data Directory: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-

## 2. Verify Input Files

The expected files are:

- `data/customer_master.csv`
- `data/customer_incremental.csv`

If these files are missing, this cell creates a small sample dataset so that the notebook still runs.

In [2]:
import pandas as pd
from datetime import date

master_path = data_dir / "customer_master.csv"
incremental_path = data_dir / "customer_incremental.csv"

if not master_path.exists() or not incremental_path.exists():
    print("Customer CSV files not found. Creating small sample files for execution safety...")
    sample_master = pd.DataFrame([
        {"customer_id":"C001", "customer_name":"Aarav Sharma", "segment":"Consumer", "country":"United States", "city":"Seattle", "state":"Washington", "postal_code":"98103", "region":"West", "first_order_date":"2016-01-10", "last_order_date":"2016-11-20", "total_orders":3, "total_sales":520.50, "record_start_date":"2016-12-31"},
        {"customer_id":"C002", "customer_name":"Priya Mehta", "segment":"Corporate", "country":"United States", "city":"New York City", "state":"New York", "postal_code":"10024", "region":"East", "first_order_date":"2016-03-15", "last_order_date":"2016-12-01", "total_orders":5, "total_sales":1220.00, "record_start_date":"2016-12-31"},
        {"customer_id":"C003", "customer_name":"Rahul Verma", "segment":"Home Office", "country":"United States", "city":"Chicago", "state":"Illinois", "postal_code":"60610", "region":"Central", "first_order_date":"2016-05-05", "last_order_date":"2016-10-09", "total_orders":2, "total_sales":300.75, "record_start_date":"2016-12-31"}
    ])
    sample_incremental = pd.DataFrame([
        {"customer_id":"C001", "customer_name":"Aarav Sharma", "segment":"Corporate", "country":"United States", "city":"San Francisco", "state":"California", "postal_code":"94109", "region":"West", "first_order_date":"2017-01-05", "last_order_date":"2017-10-10", "total_orders":4, "total_sales":800.00, "record_start_date":"2018-01-01"},
        {"customer_id":"C004", "customer_name":"Neha Kapoor", "segment":"Consumer", "country":"United States", "city":"Boston", "state":"Massachusetts", "postal_code":"02108", "region":"East", "first_order_date":"2017-04-15", "last_order_date":"2017-12-02", "total_orders":2, "total_sales":410.00, "record_start_date":"2018-01-01"}
    ])
    sample_master.to_csv(master_path, index=False)
    sample_incremental.to_csv(incremental_path, index=False)

print("Master file:", master_path)
print("Incremental file:", incremental_path)
print("Master exists:", master_path.exists())
print("Incremental exists:", incremental_path.exists())

Master file: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\data\customer_master.csv
Incremental file: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\data\customer_incremental.csv
Master exists: True
Incremental exists: True


## 3. Load Data

This loads the master and incremental customer files. In Delta Spark mode, Spark DataFrames are used. In fallback mode, pandas DataFrames are used.

In [3]:
if USE_DELTA:
    master_df = spark.read.option("header", True).option("inferSchema", True).csv(str(master_path))
    incremental_df = spark.read.option("header", True).option("inferSchema", True).csv(str(incremental_path))

    print("Master Count:", master_df.count())
    print("Incremental Count:", incremental_df.count())
    master_df.show(5, truncate=False)
    incremental_df.show(5, truncate=False)
else:
    master_pd = pd.read_csv(master_path)
    incremental_pd = pd.read_csv(incremental_path)

    print("Master Count:", len(master_pd))
    print("Incremental Count:", len(incremental_pd))
    display(master_pd.head())
    display(incremental_pd.head())

Master Count: 200
Incremental Count: 81


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,last_order_date,total_orders,total_sales,record_start_date
0,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,2015-10-04,2016-03-03,2,4433.03,2016-12-31
1,AA-10375,Allen Armold,Consumer,United States,Mesa,Arizona,85204,West,2015-02-03,2016-07-10,3,200.39,2016-12-31
2,AA-10480,Andrew Allen,Consumer,United States,Middletown,Connecticut,6457,East,2014-05-04,2014-05-04,1,27.46,2016-12-31
3,AA-10645,Anna Andreadi,Consumer,United States,Georgetown,Kentucky,40324,South,2014-12-01,2016-09-04,3,1995.74,2016-12-31
4,AB-10015,Aaron Bergman,Consumer,United States,Oklahoma City,Oklahoma,73120,Central,2014-03-07,2016-11-10,2,873.53,2016-12-31


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,last_order_date,total_orders,total_sales,record_start_date
0,AA-10375,Allen Armold,Corporate,United States,New York City,New York,10035,East,2017-09-07,2017-12-11,2,206.73,2018-01-01
1,AA-10645,Anna Andreadi,Consumer,United States,San Diego Updated,California,92105,West,2017-11-05,2017-11-05,1,12.96,2018-01-01
2,AB-10060,Adam Bellavance,Home Office,United States,Seattle,Washington,98105,East,2017-05-07,2017-11-06,3,2915.53,2018-01-01
3,AB-10105,Adrian Barton,Corporate,United States,NaN,Illinois,61701,Central,2017-06-05,2017-08-03,2,510.19,2018-01-01
4,AB-10150,Aimee Bixby,Consumer,United States,Long Beach Updated,New York,11561,East,2017-09-04,2017-09-04,1,91.96,2018-01-01


## 4. Data Cleaning

Cleaning includes:

- Trimming string columns
- Filling missing city/state/region values
- Removing duplicate customer records
- Converting date columns to proper date format

In [4]:
tracked_columns = ["customer_name", "segment", "country", "city", "state", "postal_code", "region"]
business_columns = [
    "customer_id", "customer_name", "segment", "country", "city", "state",
    "postal_code", "region", "first_order_date", "last_order_date",
    "total_orders", "total_sales", "record_start_date"
]

if USE_DELTA:
    def clean_customer_spark(df):
        cleaned = df
        for column_name, dtype in cleaned.dtypes:
            if dtype == "string":
                cleaned = cleaned.withColumn(column_name, trim(col(column_name)))

        cleaned = (
            cleaned
            .withColumn("city", when((col("city").isNull()) | (col("city") == ""), lit("Unknown")).otherwise(col("city")))
            .withColumn("state", when((col("state").isNull()) | (col("state") == ""), lit("Unknown")).otherwise(col("state")))
            .withColumn("region", when((col("region").isNull()) | (col("region") == ""), lit("Unknown")).otherwise(col("region")))
            .withColumn("first_order_date", to_date(col("first_order_date")))
            .withColumn("last_order_date", to_date(col("last_order_date")))
            .withColumn("record_start_date", to_date(col("record_start_date")))
            .dropDuplicates(["customer_id"])
            .filter(col("customer_id").isNotNull())
        )
        return cleaned

    master_clean = clean_customer_spark(master_df)
    incremental_clean = clean_customer_spark(incremental_df)

    print("Clean Master Count:", master_clean.count())
    print("Clean Incremental Count:", incremental_clean.count())
    master_clean.show(5, truncate=False)
else:
    def clean_customer_pandas(df):
        cleaned = df.copy()
        for c in cleaned.select_dtypes(include="object").columns:
            cleaned[c] = cleaned[c].astype(str).str.strip()
            cleaned[c] = cleaned[c].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})

        for c in ["city", "state", "region"]:
            cleaned[c] = cleaned[c].fillna("Unknown")

        for c in ["first_order_date", "last_order_date", "record_start_date"]:
            cleaned[c] = pd.to_datetime(cleaned[c], errors="coerce").dt.date

        cleaned = cleaned.dropna(subset=["customer_id"])
        cleaned = cleaned.drop_duplicates(subset=["customer_id"], keep="last")
        return cleaned[business_columns]

    master_clean_pd = clean_customer_pandas(master_pd)
    incremental_clean_pd = clean_customer_pandas(incremental_pd)

    print("Clean Master Count:", len(master_clean_pd))
    print("Clean Incremental Count:", len(incremental_clean_pd))
    display(master_clean_pd.head())

Clean Master Count: 200
Clean Incremental Count: 80


C:\Users\saksh\AppData\Local\Temp\ipykernel_22604\2909791384.py:37: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in cleaned.select_dtypes(include="object").columns:
C:\Users\saksh\AppData\Local\Temp\ipykernel_22604\2909791384.py:37: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_g

,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,last_order_date,total_orders,total_sales,record_start_date
0,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,2015-10-04,2016-03-03,2,4433.03,2016-12-31
1,AA-10375,Allen Armold,Consumer,United States,Mesa,Arizona,85204,West,2015-02-03,2016-07-10,3,200.39,2016-12-31
2,AA-10480,Andrew Allen,Consumer,United States,Middletown,Connecticut,6457,East,2014-05-04,2014-05-04,1,27.46,2016-12-31
3,AA-10645,Anna Andreadi,Consumer,United States,Georgetown,Kentucky,40324,South,2014-12-01,2016-09-04,3,1995.74,2016-12-31
4,AB-10015,Aaron Bergman,Consumer,United States,Oklahoma City,Oklahoma,73120,Central,2014-03-07,2016-11-10,2,873.53,2016-12-31


## 5. SCD Type 1 Implementation

SCD Type 1 keeps only the latest customer state. If a customer already exists, the old values are overwritten.

In [5]:
if USE_DELTA:
    scd1_path = str(delta_dir / "customer_scd1")

    # Clean previous Delta table output to avoid old schema conflicts during reruns
    if Path(scd1_path).exists():
        shutil.rmtree(scd1_path)

    (
        master_clean
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(scd1_path)
    )

    scd1_table = DeltaTable.forPath(spark, scd1_path)

    (
        scd1_table.alias("target")
        .merge(
            incremental_clean.alias("updates"),
            "target.customer_id = updates.customer_id"
        )
        .whenMatchedUpdate(set={
            "customer_name": "updates.customer_name",
            "segment": "updates.segment",
            "country": "updates.country",
            "city": "updates.city",
            "state": "updates.state",
            "postal_code": "updates.postal_code",
            "region": "updates.region",
            "first_order_date": "updates.first_order_date",
            "last_order_date": "updates.last_order_date",
            "total_orders": "updates.total_orders",
            "total_sales": "updates.total_sales",
            "record_start_date": "updates.record_start_date"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

    scd1_final = spark.read.format("delta").load(scd1_path)
    print("SCD1 Final Count:", scd1_final.count())
    scd1_final.orderBy("customer_id").show(10, truncate=False)
else:
    master_idx = master_clean_pd.set_index("customer_id")
    incremental_idx = incremental_clean_pd.set_index("customer_id")

    # Existing customers are overwritten, new customers are inserted
    scd1_pd = master_idx.copy()
    scd1_pd.update(incremental_idx)
    new_customers = incremental_idx.loc[~incremental_idx.index.isin(scd1_pd.index)]
    scd1_pd = pd.concat([scd1_pd, new_customers], axis=0).reset_index()

    print("SCD1 Final Count:", len(scd1_pd))
    display(scd1_pd.sort_values("customer_id").head(10))

SCD1 Final Count: 217


,customer_id,customer_name,segment,country,city,state,postal_code,region,first_order_date,last_order_date,total_orders,total_sales,record_start_date
0,AA-10315,Alex Avila,Consumer,United States,Round Rock,Texas,78664,Central,2015-10-04,2016-03-03,2,4433.03,2016-12-31
1,AA-10375,Allen Armold,Corporate,United States,New York City,New York,10035,East,2017-09-07,2017-12-11,2,206.73,2018-01-01
2,AA-10480,Andrew Allen,Consumer,United States,Middletown,Connecticut,6457,East,2014-05-04,2014-05-04,1,27.46,2016-12-31
3,AA-10645,Anna Andreadi,Consumer,United States,San Diego Updated,California,92105,West,2017-11-05,2017-11-05,1,12.96,2018-01-01
4,AB-10015,Aaron Bergman,Consumer,United States,Oklahoma City,Oklahoma,73120,Central,2014-03-07,2016-11-10,2,873.53,2016-12-31
5,AB-10060,Adam Bellavance,Home Office,United States,Seattle,Washington,98105,East,2017-05-07,2017-11-06,3,2915.53,2018-01-01
6,AB-10105,Adrian Barton,Corporate,United States,Unknown,Illinois,61701,Central,2017-06-05,2017-08-03,2,510.19,2018-01-01
7,AB-10150,Aimee Bixby,Consumer,United States,Long Beach Updated,New York,11561,East,2017-09-04,2017-09-04,1,91.96,2018-01-01
8,AB-10165,Alan Barnes,Consumer,United States,Bellevue,Washington,98006,East,2017-12-05,2017-12-05,1,39.79,2018-01-01
9,AB-10255,Alejandro Ballentine,Consumer,United States,Lorain,Ohio,44052,East,2017-06-02,2017-06-02,1,30.41,2018-01-01


## 6. SCD Type 2 Implementation

SCD Type 2 keeps historical changes. When tracked attributes change, the old row is expired and a new current row is inserted.

In [ ]:
# ---------------------------------------------------------
# SCD Type 2 Implementation
# ---------------------------------------------------------

if USE_DELTA:
    scd2_path = str(delta_dir / "customer_scd2")

    if Path(scd2_path).exists():
        shutil.rmtree(scd2_path)

    master_scd2 = (
        master_clean
        .withColumn("effective_start_date", col("record_start_date"))
        .withColumn("effective_end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn(
            "record_hash",
            sha2(concat_ws("||", *[col(c).cast("string") for c in tracked_columns]), 256)
        )
    )

    (
        master_scd2
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(scd2_path)
    )

    updates_hashed = (
        incremental_clean
        .withColumn("effective_start_date", col("record_start_date"))
        .withColumn("effective_end_date", lit(None).cast("date"))
        .withColumn("is_current", lit(True))
        .withColumn(
            "record_hash",
            sha2(concat_ws("||", *[col(c).cast("string") for c in tracked_columns]), 256)
        )
    )

    current_scd2 = (
        spark.read
        .format("delta")
        .load(scd2_path)
        .filter(col("is_current") == True)
    )

    changed_or_new = (
        updates_hashed.alias("u")
        .join(current_scd2.alias("t"), "customer_id", "left")
        .filter(
            (col("t.customer_id").isNull()) |
            (col("u.record_hash") != col("t.record_hash"))
        )
        .select("u.*")
    )

    changed_customer_ids = changed_or_new.select("customer_id").distinct()

    scd2_table = DeltaTable.forPath(spark, scd2_path)

    (
        scd2_table.alias("target")
        .merge(
            changed_customer_ids.alias("updates"),
            "target.customer_id = updates.customer_id AND target.is_current = true"
        )
        .whenMatchedUpdate(set={
            "effective_end_date": "current_date()",
            "is_current": "false"
        })
        .execute()
    )

    (
        changed_or_new
        .write
        .format("delta")
        .mode("append")
        .save(scd2_path)
    )

    scd2_final = spark.read.format("delta").load(scd2_path)

    print("SCD2 Final Count:", scd2_final.count())
    scd2_final.orderBy("customer_id", "effective_start_date").show(20, truncate=False)

else:
    # ---------------------------------------------------------
    # Pandas fallback SCD Type 2
    # ---------------------------------------------------------

    def row_hash(row):
        return "||".join(str(row[c]) for c in tracked_columns)

    # Make fresh copies so original dataframes are not affected
    scd2_pd = master_clean_pd.copy()
    incremental_pd = incremental_clean_pd.copy()

    # Convert dates properly before SCD2 processing
    scd2_pd["record_start_date"] = pd.to_datetime(
        scd2_pd["record_start_date"],
        errors="coerce"
    )

    incremental_pd["record_start_date"] = pd.to_datetime(
        incremental_pd["record_start_date"],
        errors="coerce"
    )

    # Create SCD2 columns
    scd2_pd["effective_start_date"] = scd2_pd["record_start_date"]
    scd2_pd["effective_end_date"] = pd.NaT
    scd2_pd["effective_end_date"] = pd.to_datetime(
        scd2_pd["effective_end_date"],
        errors="coerce"
    )
    scd2_pd["is_current"] = True
    scd2_pd["record_hash"] = scd2_pd.apply(row_hash, axis=1)

    for _, upd in incremental_pd.iterrows():
        customer_id = upd["customer_id"]
        upd_hash = row_hash(upd)

        current_mask = (
            (scd2_pd["customer_id"] == customer_id) &
            (scd2_pd["is_current"] == True)
        )

        if current_mask.any():
            old_hash = scd2_pd.loc[current_mask, "record_hash"].iloc[0]

            if old_hash != upd_hash:
                # Convert update date before assigning to datetime column
                update_start_date = pd.to_datetime(
                    upd["record_start_date"],
                    errors="coerce"
                )

                # Expire previous current record
                scd2_pd.loc[current_mask, "effective_end_date"] = update_start_date
                scd2_pd.loc[current_mask, "is_current"] = False

                # Insert new current record
                new_row = upd.to_dict()
                new_row.update({
                    "effective_start_date": update_start_date,
                    "effective_end_date": pd.NaT,
                    "is_current": True,
                    "record_hash": upd_hash
                })

                scd2_pd = pd.concat(
                    [scd2_pd, pd.DataFrame([new_row])],
                    ignore_index=True
                )

        else:
            # Insert completely new customer
            update_start_date = pd.to_datetime(
                upd["record_start_date"],
                errors="coerce"
            )

            new_row = upd.to_dict()
            new_row.update({
                "effective_start_date": update_start_date,
                "effective_end_date": pd.NaT,
                "is_current": True,
                "record_hash": upd_hash
            })

            scd2_pd = pd.concat(
                [scd2_pd, pd.DataFrame([new_row])],
                ignore_index=True
            )

    # Final datetime cleanup after concat
    scd2_pd["effective_start_date"] = pd.to_datetime(
        scd2_pd["effective_start_date"],
        errors="coerce"
    )

    scd2_pd["effective_end_date"] = pd.to_datetime(
        scd2_pd["effective_end_date"],
        errors="coerce"
    )

    print("SCD2 Final Count:", len(scd2_pd))

    display(
        scd2_pd
        .sort_values(["customer_id", "effective_start_date"])
        .head(20)
    )

TypeError: Invalid value '2018-01-01' for dtype 'datetime64[ns]'

## 7. Validation Checks

These checks confirm that:

- SCD Type 1 has only one row per customer
- SCD Type 2 has only one current row per customer
- SCD Type 2 contains historical rows when changes occur

In [7]:
if USE_DELTA:
    scd1_duplicate_check = (
        scd1_final
        .groupBy("customer_id")
        .agg(count("*").alias("row_count"))
        .filter(col("row_count") > 1)
    )

    scd2_current_duplicate_check = (
        scd2_final
        .filter(col("is_current") == True)
        .groupBy("customer_id")
        .agg(count("*").alias("current_row_count"))
        .filter(col("current_row_count") > 1)
    )

    historical_rows = scd2_final.filter(col("is_current") == False).count()

    print("SCD1 duplicate customer IDs:", scd1_duplicate_check.count())
    print("SCD2 duplicate current customer IDs:", scd2_current_duplicate_check.count())
    print("SCD2 historical rows:", historical_rows)
else:
    scd1_duplicates = scd1_pd[scd1_pd.duplicated("customer_id", keep=False)]
    current_counts = scd2_pd[scd2_pd["is_current"] == True].groupby("customer_id").size().reset_index(name="current_row_count")
    scd2_current_duplicates = current_counts[current_counts["current_row_count"] > 1]
    historical_rows = len(scd2_pd[scd2_pd["is_current"] == False])

    print("SCD1 duplicate customer IDs:", len(scd1_duplicates))
    print("SCD2 duplicate current customer IDs:", len(scd2_current_duplicates))
    print("SCD2 historical rows:", historical_rows)

SCD1 duplicate customer IDs: 0
SCD2 duplicate current customer IDs: 0
SCD2 historical rows: 0


## 8. Save Final Outputs

Final outputs are saved inside the `outputs/` folder for GitHub review and screenshots.

In [8]:
if USE_DELTA:
    scd1_output = output_dir / "scd1_final_csv"
    scd2_output = output_dir / "scd2_final_csv"

    if scd1_output.exists():
        shutil.rmtree(scd1_output)
    if scd2_output.exists():
        shutil.rmtree(scd2_output)

    (
        scd1_final
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(str(scd1_output))
    )

    (
        scd2_final
        .coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(str(scd2_output))
    )

    print("Spark/Delta outputs saved inside:", output_dir)
else:
    scd1_pd.to_csv(output_dir / "scd1_final.csv", index=False)
    scd2_pd.to_csv(output_dir / "scd2_final.csv", index=False)

    print("Pandas fallback outputs saved inside:", output_dir)
    print("Files created:")
    print("-", output_dir / "scd1_final.csv")
    print("-", output_dir / "scd2_final.csv")

Pandas fallback outputs saved inside: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\outputs
Files created:
- c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\outputs\scd1_final.csv
- c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\outputs\scd2_final.csv


## 9. Final Summary

SCD Type 1 is used when the latest customer information is enough and historical changes are not required.

SCD Type 2 is used when the business needs to preserve previous values and track customer changes over time.

Delta Lake is suitable for this assignment because it supports reliable table updates, merge operations, schema handling, and transactional processing.

In [9]:
summary = {
    "runtime_mode": RUNTIME_MODE,
    "project_root": str(project_root),
    "data_directory": str(data_dir),
    "delta_directory": str(delta_dir),
    "output_directory": str(output_dir)
}

for key, value in summary.items():
    print(f"{key}: {value}")

print("Week 7 Delta Lake SCD assignment notebook completed successfully.")

runtime_mode: pandas_fallback
project_root: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment
data_directory: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\data
delta_directory: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\delta_tables
output_directory: c:\Celebal Assignments\Celebal-Assignments\Week-7 Delta-Lake-Assignment\outputs
Week 7 Delta Lake SCD assignment notebook completed successfully.
